# One-time AdventureWorks Product Embeddings Load into Lakebase (pgvector)

This notebook:
1. Reads products from our synced UC table.
2. Builds an `embed_text` string.
3. Calls a Databricks Model Serving embedding endpoint.
4. Creates `search.product_embeddings` in Lakebase (if missing).
5. Upserts vectors into Lakebase.

Notes:
- We already have an OAuth token for Lakebase. This notebook uses that token as the Postgres password.
- Do not commit the token to source control.


In [0]:
# 0) Install dependencies (safe on serverless; affects only this session)
%pip install -U psycopg2-binary pgvector mlflow
%restart_python


In [0]:
# 1) CONFIG
from urllib.parse import unquote

# UC table that represents your synced product table
UC_TABLE = "mini_project.lakebase_silver_sync.product"

# Lakebase Postgres connection params
PGHOST = "ep-delicate-bread-d2dup5e0.database.us-east-1.cloud.databricks.com"
PGDATABASE = "databricks_postgres"
PGUSER = unquote("besnik.sylaj%40datasurge.com")
PGSSLMODE = "require"

# Paste your Lakebase OAuth token here (short-lived). Treat it like a password.
# TIP: You can also store it in a Databricks secret and load it instead.
LAKEBASE_OAUTH_TOKEN = "" # Redacted...going to have to turn this into a secret somehow and pass it in 

# Databricks Model Serving embedding endpoint name
# Example: "bge-large-embeddings" or whatever you created.
EMBEDDING_ENDPOINT_NAME = "databricks-bge-large-en"

# One-time load controls
LIMIT_ROWS = None   # set to an int for quick test, like 100
EMBED_BATCH_SIZE = 64
UPSERT_BATCH_SIZE = 500

In [0]:
# 2) Connect to Lakebase
import psycopg2
from pgvector.psycopg2 import register_vector

conn = psycopg2.connect(
    host=PGHOST,
    dbname=PGDATABASE,
    user=PGUSER,
    password=LAKEBASE_OAUTH_TOKEN,
    sslmode=PGSSLMODE,
)
register_vector(conn)

with conn.cursor() as cur:
    cur.execute("SELECT current_database(), current_user")
    print(cur.fetchone())

print("Connected to Lakebase.")


In [0]:
# 3) Load products from UC and build embed_text
from pyspark.sql import functions as F

df = spark.table(UC_TABLE)

# Adjust these fields to match your schema if needed.
# Keep it simple for AdventureWorks: name + a few attributes.
df_embed = (
    df.select(
        F.col("product_id").cast("long").alias("product_id"),
        F.concat_ws(
            " | ",
            F.coalesce(F.col("name").cast("string"), F.lit("")),
            F.coalesce(F.col("color").cast("string"), F.lit("")),
            F.coalesce(F.col("size").cast("string"), F.lit("")),
            F.coalesce(F.col("product_line").cast("string"), F.lit("")),
            F.coalesce(F.col("class").cast("string"), F.lit("")),
            F.coalesce(F.col("style").cast("string"), F.lit("")),
        ).alias("embed_text"),
    )
    .filter(F.col("product_id").isNotNull())
    .filter(F.length(F.col("embed_text")) > 0)
)

if LIMIT_ROWS is not None:
    df_embed = df_embed.limit(int(LIMIT_ROWS))

rows = [(r["product_id"], r["embed_text"]) for r in df_embed.collect()]
print("Rows to embed:", len(rows))
print("Sample:", rows[0] if rows else None)


In [0]:
# 4) Call the embedding endpoint
import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")

def embed_texts(text_list):
    """Returns a list of embedding vectors for the input texts."""
    resp = client.predict(
        endpoint=EMBEDDING_ENDPOINT_NAME,
        inputs={"input": text_list},
    )

    # Common response formats
    if isinstance(resp, dict):
        if "data" in resp and resp["data"] and "embedding" in resp["data"][0]:
            return [d["embedding"] for d in resp["data"]]
        if "embeddings" in resp:
            return resp["embeddings"]

    raise ValueError(f"Unexpected embedding response format: {resp}")

test = embed_texts(["hello world"])
EMBED_DIM = len(test[0])
print("Embedding dimension:", EMBED_DIM)

In [0]:
# 5) Create embeddings table + index in Lakebase (dimension must match model)
from psycopg2 import sql

with conn.cursor() as cur:
    cur.execute("CREATE SCHEMA IF NOT EXISTS search;")

    # If the table exists with the wrong dimension, drop and recreate.
    cur.execute("SELECT to_regclass('search.product_embeddings') IS NOT NULL;")
    exists = cur.fetchone()[0]

    if exists:
        cur.execute("""
            SELECT format_type(a.atttypid, a.atttypmod)
            FROM pg_attribute a
            JOIN pg_class c ON a.attrelid = c.oid
            JOIN pg_namespace n ON c.relnamespace = n.oid
            WHERE n.nspname='search'
              AND c.relname='product_embeddings'
              AND a.attname='embedding';
        """)
        col_type = cur.fetchone()[0]  # like vector(1024)
        print("Existing embedding col type:", col_type)
        if f"vector({EMBED_DIM})" not in col_type:
            print("Dimension mismatch; dropping table to recreate.")
            cur.execute("DROP TABLE search.product_embeddings;")
            exists = False

    if not exists:
        cur.execute(
            sql.SQL("""
                CREATE TABLE search.product_embeddings (
                  product_id BIGINT PRIMARY KEY,
                  embedding  vector({dim}),
                  updated_at TIMESTAMPTZ DEFAULT now()
                );
            """).format(dim=sql.Literal(EMBED_DIM))
        )

    cur.execute("""
        CREATE INDEX IF NOT EXISTS product_embeddings_hnsw
        ON search.product_embeddings
        USING hnsw (embedding vector_cosine_ops);
    """)

    conn.commit()

print("search.product_embeddings is ready.")


In [0]:
# 6) Upsert embeddings into Lakebase (one-time load)
import time
from psycopg2.extras import execute_values

#conn.rollback()

def chunk(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

upsert_sql = """
INSERT INTO search.product_embeddings (product_id, embedding)
VALUES %s
ON CONFLICT (product_id)
DO UPDATE SET
  embedding = EXCLUDED.embedding,
  updated_at = now();
"""

total = 0
start = time.time()

with conn.cursor() as cur:
    for batch in chunk(rows, EMBED_BATCH_SIZE):
        pids = [x[0] for x in batch]
        texts = [x[1] for x in batch]

        embs = embed_texts(texts)
        upsert_rows = list(zip(pids, embs))

        for up in chunk(upsert_rows, UPSERT_BATCH_SIZE):
            execute_values(cur, upsert_sql, up, page_size=len(up))

        conn.commit()
        total += len(batch)
        if total % (EMBED_BATCH_SIZE * 5) == 0:
            print(f"Upserted {total} rows")

elapsed = round(time.time() - start, 2)
print(f"DONE. Upserted {total} rows in {elapsed}s")

In [0]:
# 7) Validate row count
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM search.product_embeddings;")
    print("Embeddings row count:", cur.fetchone()[0])

conn.close()
print("Closed connection.")
